# Incertidumbre de un comité MACE sobre la NVT de BGDY a 2250 K

Evalúa los 5 modelos del comité (`models/FT_MACE_v32_c1…c5.model`) en todos los frames de la AIMD y compara la
incertidumbre del comité con el error real frente a DFT.

**Qué calcula**
- Predicción = media de los 5 modelos. Incertidumbre: σ_E = desviación típica de E/N entre modelos; por átomo,
  σ_F,i = √(Σ_c var_k F_kic); por frame, max_i σ_F,i.
- **Calibración** (Musil 2019 / Imbalzano 2021): σ_cal = k·σ, con k ajustado en una parte de los datos.
  α reescala los miembros (F_k' = F̄ + α(F_k − F̄)); k = √(α² + 1/M) es el factor de la incertidumbre de la media.
  En una trayectoria se calibra con **bloques de 100 fs** al azar (la mitad), no con frames sueltos, porque frames
  vecinos están correlacionados; la otra mitad sirve para validar.
- **Validación**: diagrama de fiabilidad (RMSE frente a RMV), Spearman σ–error, AUC para detectar frames con
  max|ΔF| > `err_thr`, umbral de σ para MD / aprendizaje activo, RMSE/RMV por bloque a lo largo del tiempo y
  comparación con los frames que tienen un enlace B–C roto.

**Carpeta esperada** (la del notebook):
```
Uncertainty quantification/
├── uq_comite_mace.ipynb
├── models/FT_MACE_v32_c1.model … FT_MACE_v32_c5.model
├── bgdy_2250K_nvt.xyz             ← del zip (o los outputs de QE en qe_outputs/, ver sección 1)
├── train_v32_filtrado.xyz         ← opcional: marca y excluye los frames que estén en el train
└── test_v32_filtrado_total.xyz    ← opcional: calibrar con el test (CALIB_XYZ)
```
Necesita `mace-torch` (probado con 0.3.16), `ase`, `numpy`, `matplotlib`; `tqdm` y `pandas` son opcionales.
Si el kernel muere sin dar error, mira la terminal desde la que lanzaste Jupyter Lab: ahí sale el mensaje
(p. ej. `OMP: Error #15` o falta de memoria).

Los resultados van a la carpeta `args.out` (CSV, JSON y PNG). Las predicciones se guardan en
`committee_preds.npz`: si cambias solo la calibración, la sección 4 se repite sin volver a evaluar.

In [ ]:
import os
# numpy (MKL) y PyTorch traen cada uno su propia librería OpenMP. En Windows/conda eso mata el kernel
# ("OMP: Error #15 ... libiomp5md.dll already initialized") en la primera operación de álgebra lineal.
# Esta variable lo evita. Tiene que estar antes de importar numpy y torch: tras cambiarla, reinicia el kernel.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import sys, re, json, glob, time, warnings
from types import SimpleNamespace
from collections import OrderedDict
import numpy as np
%matplotlib inline

# ------------------------------- Parámetros -------------------------------
MODELOS = sorted(glob.glob("models/FT_MACE_v32_c*.model"))
TRAYECTORIA = "bgdy_2000K_nvt.xyz"   # si no existe, la sección 1 la crea a partir de QE_OUTPUTS
QE_OUTPUTS = ["qe_outputs_2500_MN5_stable_a/output.1", "qe_outputs_2500_MN5_stable_a/output.2"]
TRAIN_XYZ = None    # p. ej. ["train_v32_filtrado.xyz"]: los frames que estén en el train no cuentan
CALIB_XYZ = None    # p. ej. ["test_v32_filtrado_total.xyz"]: calibra con el test y evalúa toda la trayectoria
RECALCULAR = False  # True: evalúa los modelos aunque ya existan las predicciones guardadas

args = SimpleNamespace(
    split="auto",          # auto (= bloques en una trayectoria) | bloques | tiempo | aleatorio
    calib_frac=0.5,        # fracción de la trayectoria para calibrar (si no hay CALIB_XYZ)
    block_fs=100.0,        # longitud de bloque (fs)
    stride=1,              # 1 = los 2500 frames; 5 = uno de cada 5
    seed=0,
    energy_key="energy_qe", forces_key="forces_qe", group_by="config_type",
    head=None,             # None: MACE elige la cabeza (en un fine-tuning naive solo hay una)
    device="cuda", dtype="float64",
    alpha_method="musil",  # musil | rmv | mle
    err_thr=0.2,           # eV/Å: frame "malo" si max_i |ΔF_i| > err_thr
    nbins=10, match_tol_E=0.01,
    out="uq_2000K",
)
# ---------------------------------------------------------------------------

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*weights_only.*")
import torch
if args.device == "cuda" and not torch.cuda.is_available():
    print("No hay GPU visible para PyTorch: se usa la CPU (mucho más lento; prueba stride=5)")
    args.device = "cpu"
elif args.device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(f"{len(MODELOS)} modelos:", [os.path.basename(m) for m in MODELOS])
assert len(MODELOS) >= 2, "No encuentro los modelos en models/"

## 1. Trayectoria: unir los outputs de QE (solo si falta el xyz)

Si ya tienes `bgdy_2250K_nvt.xyz` (del zip) esta sección no hace nada. Si no, une los `output.N` de pw.x:
- el primer SCF de la primera ejecución es el paso 0;
- en cada reinicio, la cabecera repite la geometría del input, pero pw.x usa la del directorio save: su primer
  SCF es el paso (primera iteración − 1) con la última `ATOMIC_POSITIONS` del output anterior;
- energía de la línea `!` (eV), fuerzas en eV/Å, CODATA 2006 como el lector de ASE; info: `step`, `time_fs`,
  `T_inst_K`, `mag_tot`, `mag_abs`, `n_BC_rotos` (B–C de la geometría inicial con d > 2.2 Å).

In [ ]:
from ase import Atoms
from ase.io import write
from ase.units import create_units

_u = create_units("2006")
RY = _u["Ry"]
BOHR = _u["Bohr"]
RY_BOHR = RY / BOHR
FLOAT = r"[-+]?\d*\.\d+(?:[eEdD][-+]?\d+)?"


def parse_output(path):
    with open(path, errors="replace") as fh:
        lines = fh.read().splitlines()
    out = dict(path=path, name=os.path.basename(path))
    nat = alat = None
    axes = []
    head_sym, head_pos = [], []
    dt_fs = None
    i = 0
    # ---- cabecera
    while i < len(lines):
        L = lines[i]
        if "number of atoms/cell" in L:
            nat = int(L.split("=")[1])
        elif "lattice parameter (alat)" in L:
            alat = float(L.split("=")[1].split()[0]) * BOHR
        elif L.strip().startswith("a(") and "=" in L and len(axes) < 3:
            axes.append([float(x) for x in re.findall(FLOAT, L.split("=")[1])][:3])
        elif "positions (alat units)" in L and not head_pos:
            for j in range(1, nat + 1):
                p = lines[i + j].split()
                head_sym.append(p[1])
                head_pos.append([float(x) for x in re.findall(FLOAT, lines[i + j].split("=")[1])][:3])
            i += nat
        elif "Time step" in L and "femto" in L:
            dt_fs = float(re.findall(FLOAT, L.split(",")[1])[0])
        elif "Self-consistent Calculation" in L or "Entering Dynamics" in L:
            break
        i += 1
    if nat is None or alat is None or len(axes) < 3:
        raise ValueError(f"{path}: cabecera incompleta")
    out["nat"] = nat
    out["cell"] = np.array(axes) * alat
    out["head_sym"] = head_sym
    out["head_pos"] = np.array(head_pos) * alat
    out["restart"] = any("Atomic positions from file used" in L for L in lines[:2000])

    # ---- cuerpo
    scfs = []          # lista de dict(E, F, mag_tot, mag_abs, it_before)
    positions = {}     # paso -> (símbolos, posiciones Å)
    temps = {}         # paso -> T
    E = None
    mag_tot = mag_abs = np.nan
    last_it = None     # última "iteration = N" vista
    first_it = None
    while i < len(lines):
        L = lines[i]
        if L.startswith("!") and "total energy" in L:
            E = float(re.findall(FLOAT, L)[0]) * RY
        elif "total magnetization" in L:
            v = re.findall(FLOAT, L)
            mag_tot = float(v[0]) if v else np.nan
        elif "absolute magnetization" in L:
            v = re.findall(FLOAT, L)
            mag_abs = float(v[0]) if v else np.nan
        elif "Forces acting on atoms" in L and E is not None:
            F = []
            j = i + 1
            while j < len(lines) and len(F) < nat:
                if "force =" in lines[j] and lines[j].strip().startswith("atom"):
                    F.append([float(x) for x in re.findall(FLOAT, lines[j].split("=")[1])][:3])
                elif F and lines[j].strip() == "":
                    break
                j += 1
            if len(F) == nat:
                scfs.append(dict(E=E, F=np.array(F) * RY_BOHR, mag_tot=mag_tot, mag_abs=mag_abs, it=last_it))
            E = None
            i = j
            continue
        elif "Entering Dynamics" in L and "iteration" in L:
            last_it = int(L.split("=")[1])
            if first_it is None:
                first_it = last_it
        elif L.startswith("ATOMIC_POSITIONS") and last_it is not None:
            scale = 1.0
            if "bohr" in L:
                scale = BOHR
            elif "alat" in L:
                scale = alat
            elif "crystal" in L:
                raise ValueError("ATOMIC_POSITIONS en crystal no soportado")
            sym, pos = [], []
            for j in range(1, nat + 1):
                p = lines[i + j].split()
                sym.append(p[0])
                pos.append([float(x) for x in p[1:4]])
            positions[last_it] = (sym, np.array(pos) * scale)
            i += nat
        elif "Time step" in L and "femto" in L:
            dt_fs = float(re.findall(FLOAT, L.split(",")[1])[0])
        elif "temperature           =" in L and last_it is not None:
            temps[last_it] = float(re.findall(FLOAT, L)[0])
        i += 1
    out.update(scfs=scfs, positions=positions, temps=temps, first_it=first_it, dt_fs=dt_fs)
    return out


def assign_steps(runs):
    """Devuelve dict paso -> frame (dict) uniendo las ejecuciones en orden."""
    runs = sorted(runs, key=lambda r: (r["first_it"] if r["first_it"] is not None else 10 ** 9))
    frames = {}
    all_pos = {}
    all_T = {}
    report = []
    for r in runs:
        it0 = r["first_it"]
        if it0 is None:
            report.append(f"  {r['name']}: sin pasos de dinámica, se salta")
            continue
        # posiciones disponibles para este output: las de ejecuciones anteriores + las propias
        pos = dict(all_pos)
        if it0 == 1:
            pos[0] = (r["head_sym"], r["head_pos"])
        elif (it0 - 1) not in pos:
            report.append(f"  {r['name']}: falta la geometría del paso {it0 - 1} (output anterior); "
                          f"se salta su primer SCF")
        pos.update(r["positions"])
        n_ok = 0
        for s in r["scfs"]:
            step = it0 - 1 if s["it"] is None else s["it"]
            if step not in pos:
                continue
            sym, xyz = pos[step]
            frames[step] = dict(sym=sym, pos=xyz, E=s["E"], F=s["F"], mag_tot=s["mag_tot"], mag_abs=s["mag_abs"],
                                T=r["temps"].get(step, all_T.get(step, np.nan)), src=r["name"], cell=r["cell"],
                                dt=r["dt_fs"])
            n_ok += 1
        all_pos.update(r["positions"])
        all_T.update(r["temps"])
        last = max(s["it"] for s in r["scfs"] if s["it"] is not None) if r["scfs"] else None
        report.append(f"  {r['name']}: iteraciones {it0}–{max(r['positions']) if r['positions'] else '?'}, "
                      f"{len(r['scfs'])} SCF completos → pasos {it0 - 1}–{last}, {n_ok} usados"
                      f"{' (reinicio)' if r['restart'] else ''}")
    return frames, report


def convert(outputs, out, config_type="aimd", stride=1, bc_enlace=1.85, bc_rotura=2.2):
    """Une los outputs y escribe el extxyz. Devuelve la lista de Atoms."""
    runs = [parse_output(p) for p in outputs]
    nat = {r["nat"] for r in runs}
    if len(nat) != 1:
        raise ValueError(f"Los outputs tienen distinto número de átomos: {nat}")
    frames, report = assign_steps(runs)
    print("\n".join(report))
    steps = sorted(frames)
    if not steps:
        raise ValueError("No hay ningún SCF completo")
    missing = sorted(set(range(steps[0], steps[-1] + 1)) - set(steps))
    print(f"{len(steps)} pasos con energía y fuerzas: {steps[0]}–{steps[-1]}; "
          f"huecos: {len(missing)}{' ' + str(missing[:10]) if missing else ''}")
    dts = {round(f["dt"], 6) for f in frames.values() if f["dt"]}
    if len(dts) != 1:
        print(f"  aviso: pasos de tiempo distintos entre outputs: {dts}")
    dt = sorted(dts)[0]

    # diagnóstico de las uniones: salto de E respecto a la mediana de |ΔE| entre pasos
    E = np.array([frames[s]["E"] for s in steps])
    dE = np.abs(np.diff(E))
    med = np.median(dE)
    srcs = [frames[s]["src"] for s in steps]
    for k in range(1, len(steps)):
        if srcs[k] != srcs[k - 1]:
            print(f"  unión {srcs[k - 1]} → {srcs[k]} en el paso {steps[k]}: |ΔE| = {dE[k - 1]:.3f} eV "
                  f"= {dE[k - 1] / med:.2f} × mediana ({med:.3f} eV)")

    # enlaces B–C de la geometría inicial
    f0 = frames[steps[0]]
    cell = f0["cell"]
    inv = np.linalg.inv(cell)
    sym0 = np.array(f0["sym"])
    iB, iC = np.where(sym0 == "B")[0], np.where(sym0 == "C")[0]

    def mic_d(pos, a, b):
        d = pos[b][None, :, :] - pos[a][:, None, :]
        fr = d @ inv
        fr[..., :2] -= np.round(fr[..., :2])
        return np.linalg.norm(fr @ cell, axis=-1)

    if len(iB) and len(iC):
        d0 = mic_d(f0["pos"], iB, iC)
        bonds = np.argwhere(d0 < bc_enlace)
        print(f"  {len(bonds)} enlaces B–C en el paso {steps[0]} (d < {bc_enlace} Å)")
    else:
        bonds = np.zeros((0, 2), int)

    atoms_list = []
    for s in steps[::stride]:
        f = frames[s]
        if list(f["sym"]) != list(f0["sym"]):
            raise ValueError(f"El orden de especies cambia en el paso {s}")
        a = Atoms(f["sym"], positions=f["pos"], cell=f["cell"], pbc=True)
        a.wrap()
        n_rot = 0
        if len(bonds):
            d = mic_d(f["pos"], iB, iC)[bonds[:, 0], bonds[:, 1]]
            n_rot = int((d > bc_rotura).sum())
        a.info.update(energy_qe=f["E"], config_type=config_type, step=int(s), time_fs=float(s * dt),
                      T_inst_K=f["T"], mag_tot=f["mag_tot"], mag_abs=f["mag_abs"], src=f["src"],
                      n_BC_rotos=n_rot)
        a.arrays["forces_qe"] = f["F"]
        atoms_list.append(a)
    write(out, atoms_list, format="extxyz")
    Fmax = max(np.abs(a.arrays["forces_qe"]).max() for a in atoms_list)
    nrot = np.array([a.info["n_BC_rotos"] for a in atoms_list])
    mag = np.array([a.info["mag_tot"] for a in atoms_list])
    print(f"Escrito {out}: {len(atoms_list)} frames (stride {stride}), dt = {dt} fs, "
          f"t = 0–{steps[-1] * dt / 1000:.3f} ps")
    print(f"  E/át: {E.min() / len(sym0):.4f} a {E.max() / len(sym0):.4f} eV; |F| máx {Fmax:.2f} eV/Å; "
          f"mag_tot {np.nanmin(mag):.2f} a {np.nanmax(mag):.2f} μB; "
          f"B–C rotos (d > {bc_rotura} Å): máx {nrot.max()}, en {np.mean(nrot > 0) * 100:.0f} % de los frames")
    return atoms_list

In [ ]:
if os.path.exists(TRAYECTORIA):
    print(f"Se usa {TRAYECTORIA}")
else:
    convert(QE_OUTPUTS, TRAYECTORIA, config_type="aimd_2250K_nvt")

## 2. Funciones

No hace falta tocarlas: lectura, evaluación del comité, estadística, calibración y figuras.

In [ ]:
META = ['time_fs', 'step', 'mag_tot', 'mag_abs', 'n_BC_rotos', 'T_inst_K']


def _ref_energy(atoms, key):
    if key in atoms.info:
        return float(atoms.info[key])
    if atoms.calc is not None and key in ("energy", "free_energy"):
        try:
            return float(atoms.get_potential_energy())
        except Exception:
            pass
    raise KeyError(f"no encuentro la energía '{key}' en info ni en el calculador")


def _ref_forces(atoms, key):
    if key in atoms.arrays:
        return np.asarray(atoms.arrays[key], dtype=float)
    if atoms.calc is not None and key == "forces":
        try:
            return np.asarray(atoms.get_forces(), dtype=float)
        except Exception:
            pass
    raise KeyError(f"no encuentro las fuerzas '{key}' en arrays ni en el calculador")


def _meta(at, key):
    try:
        v = float(at.info[key])
        return v if np.isfinite(v) else np.nan
    except (KeyError, TypeError, ValueError):
        return np.nan


def read_sets(paths, energy_key, forces_key, group_by, tag, stride=1):
    from ase.io import read

    frames = []
    for p in paths:
        stem = os.path.splitext(os.path.basename(p))[0]
        n_skip = n_in = 0
        for idx, at in enumerate(read(p, ":")):
            if str(at.info.get("config_type", "")) == "TOTAL":
                n_skip += 1
                continue
            if stride > 1 and idx % stride:
                continue
            g = at.info.get(group_by, None)
            fr = dict(atoms=at, file=stem, idx=idx, group=str(g) if g is not None else stem, src=tag,
                      E_ref=_ref_energy(at, energy_key), F_ref=_ref_forces(at, forces_key), in_train=False)
            for k in META:
                fr[k] = _meta(at, k)
            frames.append(fr)
            n_in += 1
        msg = f"  {p}: {n_in} configuraciones"
        if stride > 1:
            msg += f" (stride {stride})"
        if n_skip:
            msg += f"; {n_skip} con config_type=TOTAL saltadas"
        print(msg)
    return frames


def mark_in_train(frames, train_paths, energy_key, tol_E, tol_pos=0.01):
    """Marca in_train=True en los frames que coinciden con una configuración del train."""
    from ase.io import read

    train = []
    for p in train_paths:
        for at in read(p, ":"):
            try:
                train.append((len(at), _ref_energy(at, energy_key), at))
            except KeyError:
                pass
    if not train:
        print("  --train_xyz: ninguna configuración con energía; no se marca nada")
        return
    nat_t = np.array([t[0] for t in train])
    e_t = np.array([t[1] for t in train])
    n = 0
    for fr in frames:
        a = fr["atoms"]
        cand = np.where((nat_t == len(a)) & (np.abs(e_t - fr["E_ref"]) < tol_E))[0]
        for c in cand:
            b = train[c][2]
            if b.get_chemical_symbols() != a.get_chemical_symbols():
                continue
            d = b.positions - a.positions
            try:
                cell = a.cell.array
                f = d @ np.linalg.inv(cell)
                f[:, a.pbc] -= np.round(f[:, a.pbc])
                d = f @ cell
            except np.linalg.LinAlgError:
                pass
            if np.abs(d).max() < tol_pos:
                fr["in_train"] = True
                n += 1
                break
    print(f"  --train_xyz: {len(train)} configuraciones leídas; {n} de las evaluadas están en el train "
          f"y quedan fuera de calibración y evaluación")


def evaluate(models, frames, device, dtype, head):
    from mace.calculators import MACECalculator
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = None

    M, N = len(models), len(frames)
    nat = np.array([len(f["atoms"]) for f in frames])
    off = np.concatenate([[0], np.cumsum(nat)])
    E = np.zeros((M, N))
    F = np.zeros((M, off[-1], 3))
    for k, mp in enumerate(models):
        kw = dict(model_paths=mp, device=device, default_dtype=dtype)
        if head:
            kw["head"] = head
        calc = MACECalculator(**kw)
        t0 = time.time()
        it = enumerate(frames)
        if tqdm is not None:
            it = tqdm(it, total=N, desc=f"modelo {k + 1}/{M}", leave=False)
        for n, fr in it:
            a = fr["atoms"].copy()
            a.calc = calc
            E[k, n] = a.get_potential_energy()
            F[k, off[n]:off[n + 1]] = a.get_forces()
        print(f"  modelo {k + 1}/{M} ({os.path.basename(mp)}): {N} conf. en {time.time() - t0:.0f} s")
        del calc
    return E, F


def save_preds(path, models, frames, E, F):
    nat = np.array([len(f["atoms"]) for f in frames])
    extra = {k: np.array([f[k] for f in frames], float) for k in META}
    np.savez_compressed(
        path,
        models=np.array([os.path.basename(m) for m in models]),
        E_pred=E,
        F_pred=F,
        E_ref=np.array([f["E_ref"] for f in frames]),
        F_ref=np.concatenate([f["F_ref"] for f in frames]),
        nat=nat,
        fname=np.array([f["file"] for f in frames]),
        idx=np.array([f["idx"] for f in frames]),
        group=np.array([f["group"] for f in frames]),
        src=np.array([f["src"] for f in frames]),
        in_train=np.array([f["in_train"] for f in frames]),
        species=np.concatenate([f["atoms"].get_chemical_symbols() for f in frames]),
        **extra,
    )


def load_preds(path):
    P = dict(np.load(path, allow_pickle=False))
    N = P["E_pred"].shape[1]
    for k in META:                      # npz de versiones anteriores
        P.setdefault(k, np.full(N, np.nan))
    P.setdefault("in_train", np.zeros(N, bool))
    return P

In [ ]:
def rankdata(x):
    x = np.asarray(x, float)
    order = np.argsort(x, kind="mergesort")
    r = np.empty(len(x))
    r[order] = np.arange(1, len(x) + 1)
    xs = x[order]
    i = 0
    while i < len(xs):                   # empates: rango medio
        j = i
        while j + 1 < len(xs) and xs[j + 1] == xs[i]:
            j += 1
        if j > i:
            r[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return r


def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 3 or np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan
    ra, rb = rankdata(a), rankdata(b)
    ra -= ra.mean()
    rb -= rb.mean()
    den = np.sqrt((ra ** 2).sum() * (rb ** 2).sum())
    return float((ra * rb).sum() / den) if den > 0 else np.nan


def auc(score, label):
    """AUC (Mann-Whitney): prob. de que una conf. mala tenga más sigma que una buena."""
    label = np.asarray(label, bool)
    npos, nneg = label.sum(), (~label).sum()
    if npos == 0 or nneg == 0:
        return np.nan
    r = rankdata(score)
    return float((r[label].sum() - npos * (npos + 1) / 2) / (npos * nneg))


def youden(score, label):
    """Umbral que maximiza TPR - FPR. Devuelve (umbral, TPR, FPR)."""
    label = np.asarray(label, bool)
    if len(label) == 0 or label.all() or (~label).all():
        return np.nan, np.nan, np.nan
    best = (-1, np.nan, np.nan, np.nan)
    for t in np.unique(score):
        pred = score >= t
        tpr = (pred & label).sum() / label.sum()
        fpr = (pred & ~label).sum() / (~label).sum()
        if tpr - fpr > best[0]:
            best = (tpr - fpr, t, tpr, fpr)
    return best[1], best[2], best[3]


def alpha_fit(err2, var, d, M):
    """err2 = |e|^2 (error de la media del comité), var = sigma^2 (suma de d componentes, ddof=1).

    Modelo (Musil 2019): miembros = mu + s z ;  DFT = mu + alpha s z'.
      alpha: factor para reescalar los miembros, F_k' = F_media + alpha (F_k - F_media)
      k = sqrt(alpha^2 + 1/M): factor de la incertidumbre de la PREDICCIÓN (media), sigma_cal = k sigma
    Devuelve {método: (alpha, k)}."""
    ok = var > 0
    err2, var = err2[ok], var[ok]
    if len(err2) == 0:
        return {m: (np.nan, np.nan) for m in ("musil", "rmv", "mle")}
    r = np.mean(err2 / var)
    nu = d * (M - 1)
    a2 = dict(
        musil=(nu - 2) / nu * r - 1.0 / M if nu > 2 else np.nan,   # insesgado si sigma^2 sale de M modelos
        rmv=err2.sum() / var.sum() - 1.0 / M,                      # iguala RMSE y RMV globales
        mle=r - 1.0 / M,                                           # media de cocientes, sin corregir
    )
    out = {}
    for key, v in a2.items():
        out[key] = (float(np.sqrt(v)) if v > 0 else np.nan,
                    float(np.sqrt(v + 1.0 / M)) if v + 1.0 / M > 0 else np.nan)
    return out


def reliability(sig, err2, nbins=10, min_per_bin=5):
    """Intervalos de igual población en sigma. Devuelve RMV, RMSE, n por intervalo y ENCE."""
    if len(sig) < 2 * min_per_bin:
        return None
    nb = int(min(nbins, len(sig) // min_per_bin))
    edges = np.quantile(sig, np.linspace(0, 1, nb + 1))
    which = np.clip(np.searchsorted(edges, sig, side="right") - 1, 0, nb - 1)
    rmv, rmse, cnt = [], [], []
    for b in range(nb):
        m = which == b
        if m.sum() == 0:
            continue
        rmv.append(np.sqrt(np.mean(sig[m] ** 2)))
        rmse.append(np.sqrt(np.mean(err2[m])))
        cnt.append(int(m.sum()))
    rmv, rmse = np.array(rmv), np.array(rmse)
    ence = float(np.mean(np.abs(rmv - rmse) / rmv))
    return dict(rmv=rmv, rmse=rmse, n=np.array(cnt), ence=ence)


def choose_calib(P, use, args):
    src, group, fname, t = P["src"], P["group"], P["fname"], P["time_fs"]
    N = len(src)
    if (src == "calib").any():
        return (src == "calib") & use, f"{((src == 'calib') & use).sum()} configuraciones de --calib"
    first_name = fname[src == "eval"][0]
    first = (fname == first_name) & use
    has_t = first.any() and np.isfinite(t[first]).all()
    mode = args.split
    if mode == "auto":
        mode = "bloques" if has_t else "aleatorio"
    if mode in ("bloques", "tiempo") and not has_t:
        print(f"  aviso: '{first_name}' no tiene time_fs en todos los frames; --split {mode} → aleatorio")
        mode = "aleatorio"
    rng = np.random.default_rng(args.seed)
    is_cal = np.zeros(N, bool)
    if mode == "aleatorio":
        for g in np.unique(group[first]):
            ids = np.where(first & (group == g))[0]
            rng.shuffle(ids)
            is_cal[ids[: int(round(args.calib_frac * len(ids)))]] = True
        desc = "configuraciones al azar por grupo"
    elif mode == "bloques":
        bid = np.floor(np.where(np.isfinite(t), t, 0) / args.block_fs).astype(int)
        keys = sorted({(group[i], bid[i]) for i in np.where(first)[0]})
        nsel = int(round(args.calib_frac * len(keys)))
        sel = {keys[i] for i in rng.permutation(len(keys))[:nsel]}
        is_cal = first & np.array([(group[i], bid[i]) in sel for i in range(N)])
        desc = f"{nsel} de {len(keys)} bloques de {args.block_fs:g} fs al azar"
    else:  # tiempo
        for g in np.unique(group[first]):
            m = first & (group == g)
            tq = np.quantile(t[m], args.calib_frac)
            is_cal |= m & (t <= tq)
        desc = f"primer {100 * args.calib_frac:.0f} % del tiempo"
    return is_cal, (f"{is_cal.sum()} de {first.sum()} configuraciones de '{first_name}' "
                    f"(--split {mode}: {desc}; semilla {args.seed})")


def analyse(P, args):
    E, F = P["E_pred"], P["F_pred"]
    M, N = E.shape
    nat = P["nat"]
    off = np.concatenate([[0], np.cumsum(nat)])
    atom_conf = np.repeat(np.arange(N), nat)
    group = P["group"]
    use = ~P["in_train"].astype(bool)

    is_cal, desc = choose_calib(P, use, args)
    print(f"Calibración: {desc}")
    ev = use & ~is_cal
    print(f"Evaluación: {ev.sum()} configuraciones" + (f" (fuera {(~use).sum()} del train)" if (~use).any() else ""))
    if is_cal.sum() == 0 or ev.sum() == 0:
        sys.exit("No quedan configuraciones para calibrar o para evaluar")
    is_cal_atom = is_cal[atom_conf]

    # --- energía por átomo (eV/át)
    e_k = E / nat
    e_mean = e_k.mean(0)
    e_var = e_k.var(0, ddof=1)
    e_ref = P["E_ref"] / nat
    e_err = e_mean - e_ref

    # --- fuerzas
    F_mean = F.mean(0)
    f_var = F.var(0, ddof=1).sum(1)             # varianza total por átomo (eV/Å)^2
    f_err = F_mean - P["F_ref"]
    f_err2 = (f_err ** 2).sum(1)

    aF = alpha_fit(f_err2[is_cal_atom], f_var[is_cal_atom], d=3, M=M)
    aE = alpha_fit(e_err[is_cal] ** 2, e_var[is_cal], d=1, M=M)
    (alphaF, kF), (alphaE, kE) = aF[args.alpha_method], aE[args.alpha_method]
    if not np.isfinite(kF):
        alphaF, kF = aF["rmv"]
    if not np.isfinite(kE):
        alphaE, kE = aE["rmv"]
    fmt = lambda a: " · ".join(f"{m} {a[m][1]:.3f}" for m in ("musil", "rmv", "mle"))
    print(f"factor k (sigma_cal = k sigma) fuerzas: {fmt(aF)}  -> usado {kF:.3f} (alpha miembros {alphaF:.3f})")
    print(f"factor k (sigma_cal = k sigma) energía: {fmt(aE)}  -> usado {kE:.3f} (alpha miembros {alphaE:.3f})")
    if is_cal.sum() < 100:
        print(f"  (aviso: calibración de la energía con solo {is_cal.sum()} configuraciones: incierta)")
    if (P["time_fs"][is_cal] == P["time_fs"][is_cal]).any():
        print("  (con frames de MD la energía de frames vecinos está correlacionada: la calibración de E "
              "vale menos de lo que sugiere el número de frames)")
    if aF["musil"][1] > 0 and aF["rmv"][1] / aF["musil"][1] > 1.5:
        print("  (aviso: rmv y musil difieren mucho: el error no es proporcional a sigma en todo el rango, "
              "p. ej. unas pocas configuraciones con errores muy grandes; mira el diagrama de fiabilidad)")

    f_sig = np.sqrt(f_var)
    f_sig_cal = kF * f_sig
    e_sig = np.sqrt(e_var)
    e_sig_cal = kE * e_sig

    # --- por configuración
    max_sig = np.array([f_sig[off[n]:off[n + 1]].max() for n in range(N)])
    max_err = np.array([np.sqrt(f_err2[off[n]:off[n + 1]]).max() for n in range(N)])
    rows = []
    for n in range(N):
        s = slice(off[n], off[n + 1])
        rows.append(OrderedDict(
            file=P["fname"][n], idx=int(P["idx"][n]), group=group[n],
            uso="train" if not use[n] else ("calibracion" if is_cal[n] else "evaluacion"),
            step=P["step"][n], time_fs=P["time_fs"][n], n_atoms=int(nat[n]),
            E_ref_eV_at=e_ref[n], E_comite_eV_at=e_mean[n],
            err_E_meV_at=1e3 * e_err[n], sigma_E_meV_at=1e3 * e_sig[n], sigma_E_cal_meV_at=1e3 * e_sig_cal[n],
            rmse_F_meV_A=1e3 * np.sqrt(np.mean(f_err[s] ** 2)),
            max_errF_meV_A=1e3 * max_err[n],
            max_sigF_meV_A=1e3 * max_sig[n], mean_sigF_meV_A=1e3 * f_sig[s].mean(),
            max_sigF_cal_meV_A=1e3 * kF * max_sig[n],
            n_BC_rotos=P["n_BC_rotos"][n], mag_tot=P["mag_tot"][n], mag_abs=P["mag_abs"][n],
        ))

    # --- umbral para MD / aprendizaje activo (en el conjunto de calibración, sigma SIN calibrar)
    bad = max_err > args.err_thr
    thr, tpr, fpr = youden(max_sig[is_cal], bad[is_cal])

    # --- resumen por grupo (solo evaluación) + global
    groups = list(OrderedDict.fromkeys(group[ev]))
    summ, rel = [], OrderedDict()
    for g in groups + ["TODO"]:
        cm = ev & ((group == g) if g != "TODO" else True)
        if cm.sum() == 0:
            continue
        am = cm[atom_conf]
        rm_member = np.sqrt(((F[:, am] - P["F_ref"][am][None]) ** 2).mean(axis=(1, 2)))
        r = OrderedDict(
            grupo=g, n_conf=int(cm.sum()), n_atoms=int(am.sum()),
            RMSE_E_meV_at=1e3 * np.sqrt(np.mean(e_err[cm] ** 2)),
            RMS_sigma_E_cal_meV_at=1e3 * np.sqrt(np.mean(e_sig_cal[cm] ** 2)),
            RMSE_F_comite_meV_A=1e3 * np.sqrt(np.mean(f_err[am] ** 2)),
            RMSE_F_miembros_meV_A=f"{1e3 * rm_member.mean():.1f} ± {1e3 * rm_member.std():.1f}",
            RMSE_Fvec_meV_A=1e3 * np.sqrt(np.mean(f_err2[am])),
            RMV_F_cal_meV_A=1e3 * np.sqrt(np.mean(f_sig_cal[am] ** 2)),
            rho_atomo=spearman(f_sig[am], f_err2[am]),
            rho_conf=spearman(max_sig[cm], max_err[cm]),
            frac_malas=float(bad[cm].mean()),
            AUC=auc(max_sig[cm], bad[cm]),
            frac_marcadas=float((max_sig[cm] >= thr).mean()) if np.isfinite(thr) else np.nan,
        )
        rl = reliability(f_sig_cal[am], f_err2[am], nbins=args.nbins)
        r["ENCE_F"] = rl["ence"] if rl else np.nan
        rel[g] = rl
        summ.append(r)
    relE = reliability(e_sig_cal[ev], e_err[ev] ** 2, nbins=min(args.nbins, 6), min_per_bin=4)

    cal = dict(
        M=int(M), models=[str(m) for m in P["models"]], alpha_method=args.alpha_method,
        calibracion=desc, k_F=kF, k_E=kE, alpha_F=alphaF, alpha_E=alphaE,
        todos_F={m: dict(alpha=v[0], k=v[1]) for m, v in aF.items()},
        todos_E={m: dict(alpha=v[0], k=v[1]) for m, v in aE.items()},
        n_calib_conf=int(is_cal.sum()), n_calib_atoms=int(is_cal_atom.sum()), n_eval_conf=int(ev.sum()),
        n_en_train=int((~use).sum()),
        err_thr_eV_A=args.err_thr,
        umbral_max_sigmaF_sin_calibrar_eV_A=None if not np.isfinite(thr) else float(thr),
        umbral_max_sigmaF_convenio_MACECalculator_eV_A=None if not np.isfinite(thr)
        else float(thr * np.sqrt((M - 1) / M)),
        umbral_TPR=None if not np.isfinite(tpr) else float(tpr),
        umbral_FPR=None if not np.isfinite(fpr) else float(fpr),
        ENCE_E=relE["ence"] if relE else None,
        nota="sigma = std del comité (ddof=1). Incertidumbre de la predicción (media): sigma_cal = k sigma. "
             "Para reescalar los miembros (p. ej. propagar a propiedades): F_k' = F_media + alpha (F_k - F_media).",
    )
    D = dict(max_sig=max_sig, max_err=max_err, bad=bad, is_cal=is_cal, ev=ev, use=use, group=group, thr=thr,
             kF=kF, kE=kE, e_err=e_err, e_sig=e_sig, f_err2=f_err2, f_sig=f_sig, atom_conf=atom_conf)
    return rows, summ, cal, rel, relE, D


def trajectory_stats(P, D, args):
    """Para cada archivo con time_fs: estadística temporal. Devuelve {archivo: dict}."""
    out = OrderedDict()
    t_all = P["time_fs"]
    for fn in OrderedDict.fromkeys(P["fname"]):
        m = P["fname"] == fn
        if m.sum() < 20 or not np.isfinite(t_all[m]).all():
            continue
        idx = np.where(m)[0]
        idx = idx[np.argsort(t_all[idx])]
        t = t_all[idx]
        ev, use = D["ev"][idx], D["use"][idx]
        # RMSE / RMV por bloque (frames fuera del train)
        bid = np.floor(t / args.block_fs).astype(int)
        blocks = []
        for b in np.unique(bid):
            sel = idx[(bid == b) & use]
            if len(sel) == 0:
                continue
            am = np.isin(D["atom_conf"], sel)
            rmse = np.sqrt(np.mean(D["f_err2"][am]))
            rmv = D["kF"] * np.sqrt(np.mean(D["f_sig"][am] ** 2))
            blocks.append((b * args.block_fs, (b + 1) * args.block_fs, rmse / rmv,
                           bool(D["is_cal"][sel].any()), rmse, rmv))
        st = OrderedDict(n=int(m.sum()), n_eval=int(ev.sum()), t_ps=(t[0] / 1e3, t[-1] / 1e3))
        e = idx[ev]
        st["rho_conf_eval"] = spearman(D["max_sig"][e], D["max_err"][e])
        st["RMSE_RMV_bloques_min_max"] = (min(b[2] for b in blocks), max(b[2] for b in blocks)) if blocks else None
        nbc = P["n_BC_rotos"][e]
        if np.isfinite(nbc).all() and (nbc > 0).any() and (nbc == 0).any():
            r = nbc > 0
            st["BC_roto"] = OrderedDict(
                n_frames_roto=int(r.sum()),
                max_errF_meV_A=(1e3 * D["max_err"][e][~r].mean(), 1e3 * D["max_err"][e][r].mean()),
                max_sigF_cal_meV_A=(1e3 * D["kF"] * D["max_sig"][e][~r].mean(),
                                    1e3 * D["kF"] * D["max_sig"][e][r].mean()),
                AUC_sigma_detecta_roto=auc(D["max_sig"][e], r),
            )
        mag = np.abs(P["mag_tot"][e])
        if np.isfinite(mag).all() and mag.max() > 0.1:
            st["rho_err_vs_|mag|"] = spearman(mag, D["max_err"][e])
            st["rho_sigma_vs_|mag|"] = spearman(mag, D["max_sig"][e])
        st["_idx"], st["_blocks"] = idx, blocks
        out[fn] = st
    return out

In [ ]:
def write_csv(path, rows):
    import csv

    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        for r in rows:
            w.writerow({k: (f"{v:.6g}" if isinstance(v, (float, np.floating)) else v) for k, v in r.items()})


def print_table(summ):
    cols = [("grupo", "grupo", "{}"), ("n_conf", "n", "{:d}"),
            ("RMSE_F_comite_meV_A", "RMSE F", "{:.1f}"), ("RMSE_F_miembros_meV_A", "miembros", "{}"),
            ("RMSE_Fvec_meV_A", "RMSE|dF|", "{:.1f}"), ("RMV_F_cal_meV_A", "RMV cal", "{:.1f}"),
            ("rho_atomo", "rho át", "{:.2f}"), ("rho_conf", "rho conf", "{:.2f}"),
            ("AUC", "AUC", "{:.2f}"), ("ENCE_F", "ENCE", "{:.2f}"),
            ("RMSE_E_meV_at", "RMSE E", "{:.2f}"), ("RMS_sigma_E_cal_meV_at", "sigE cal", "{:.2f}")]
    cells = [[h for _, h, _ in cols]]
    for r in summ:
        row = []
        for k, _, f in cols:
            v = r[k]
            row.append("—" if isinstance(v, float) and not np.isfinite(v) else f.format(v))
        cells.append(row)
    w = [max(len(c[i]) for c in cells) for i in range(len(cols))]
    for j, c in enumerate(cells):
        print("  ".join(s.rjust(w[i]) if i else s.ljust(w[i]) for i, s in enumerate(c)))
        if j == 0:
            print("  ".join("-" * x for x in w))
    print("(fuerzas en meV/Å, energía en meV/át; RMSE F por componente, RMSE|dF| y RMV por átomo)")


def print_traj(tst, kF):
    for fn, st in tst.items():
        print(f"\nTrayectoria '{fn}': {st['n']} frames, {st['t_ps'][0]:.3f}–{st['t_ps'][1]:.3f} ps, "
              f"{st['n_eval']} en evaluación")
        print(f"  Spearman(max sigma, max|dF|) por frame: {st['rho_conf_eval']:.2f}")
        if st["RMSE_RMV_bloques_min_max"]:
            lo, hi = st["RMSE_RMV_bloques_min_max"]
            print(f"  RMSE/RMV por bloque: {lo:.2f} a {hi:.2f} (1 = calibrado; > 1 = el comité se fía demasiado)")
        if "BC_roto" in st:
            b = st["BC_roto"]
            print(f"  Frames con un B–C roto: {b['n_frames_roto']}. max|dF| medio sin/con: "
                  f"{b['max_errF_meV_A'][0]:.0f} / {b['max_errF_meV_A'][1]:.0f} meV/Å; "
                  f"k·max sigma sin/con: {b['max_sigF_cal_meV_A'][0]:.0f} / {b['max_sigF_cal_meV_A'][1]:.0f} meV/Å; "
                  f"AUC de sigma para detectarlos: {b['AUC_sigma_detecta_roto']:.2f}")
        if "rho_err_vs_|mag|" in st:
            print(f"  Spearman con |mag_tot|: error {st['rho_err_vs_|mag|']:.2f}, sigma {st['rho_sigma_vs_|mag|']:.2f}")


SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
INK, INK2, MUTED, GRID, SURF, CALBG = '#0b0b0b', '#52514e', '#9a9893', '#e4e3df', '#fcfcfb', '#ecebe7'


def _style(ax):
    ax.set_facecolor(SURF)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(INK2)
        ax.spines[s].set_linewidth(0.8)
    ax.tick_params(colors=INK2, labelsize=8)
    ax.grid(True, color=GRID, lw=0.6)
    ax.set_axisbelow(True)


def _in_notebook():
    return "ipykernel" in sys.modules


def _mpl():
    try:
        import matplotlib
        if not _in_notebook():
            matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"Sin figuras ({exc})")
        return None
    plt.rcParams.update({"font.size": 9, "axes.labelcolor": INK, "axes.titlecolor": INK,
                         "figure.facecolor": "white"})
    return plt


def _finish(plt, fig, path):
    """Guarda la figura; en Jupyter además la muestra."""
    fig.savefig(path, dpi=170)
    if _in_notebook():
        plt.show()
    else:
        plt.close(fig)


def plots(out, rel, relE, D, err_thr):
    plt = _mpl()
    if plt is None:
        return
    groups = [g for g in rel if g != "TODO" and rel[g] is not None]
    if len(groups) > len(SERIES):
        print(f"  {len(groups)} grupos: en el diagrama de fiabilidad solo se dibujan los {len(SERIES)} primeros y TODO")
        groups = groups[: len(SERIES)]

    # --- Figura 1: diagrama de fiabilidad (fuerzas por átomo | energía por conf.)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.4))
    ax = axs[0]
    _style(ax)
    allv = []
    for i, g in enumerate(groups):
        r = rel[g]
        ax.plot(1e3 * r["rmv"], 1e3 * r["rmse"], "-o", color=SERIES[i], lw=2, ms=4.5, mec="white", mew=1, label=g)
        allv += list(1e3 * r["rmv"]) + list(1e3 * r["rmse"])
    if rel.get("TODO") and len(groups) > 1:
        r = rel["TODO"]
        ax.plot(1e3 * r["rmv"], 1e3 * r["rmse"], "--", color=INK2, lw=1.5, label="todo (evaluación)")
        allv += list(1e3 * r["rmv"]) + list(1e3 * r["rmse"])
    if allv:
        lo, hi = min(allv) * 0.8, max(allv) * 1.25
        ax.plot([lo, hi], [lo, hi], color=MUTED, lw=1, zorder=0)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel("RMV: σ calibrada por átomo (meV/Å)")
    ax.set_ylabel("RMSE de |ΔF| por átomo (meV/Å)")
    ax.set_title("Fuerzas: diagrama de fiabilidad", loc="left", fontsize=10)
    ax.text(0.98, 0.03, "sobre la diagonal = calibrado\npor encima = demasiado confiado",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=7.5, color=INK2)
    ax.legend(frameon=False, fontsize=7.5, loc="upper left")

    ax = axs[1]
    _style(ax)
    if relE is not None:
        ax.plot(1e3 * relE["rmv"], 1e3 * relE["rmse"], "-o", color=SERIES[0], lw=2, ms=4.5, mec="white", mew=1)
        v = 1e3 * np.concatenate([relE["rmv"], relE["rmse"]])
        lo, hi = v.min() * 0.8, v.max() * 1.25
        ax.plot([lo, hi], [lo, hi], color=MUTED, lw=1, zorder=0)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    else:
        ax.text(0.5, 0.5, "pocas configuraciones", ha="center", transform=ax.transAxes, color=INK2)
    ax.set_xlabel("RMV: σ_E calibrada (meV/át)")
    ax.set_ylabel("RMSE de E (meV/át)")
    ax.set_title("Energía por configuración (evaluación)", loc="left", fontsize=10)
    fig.tight_layout()
    _finish(plt, fig, os.path.join(out, "uq_fiabilidad.png"))

    # --- Figura 2: por configuración, max sigma (calibrada) frente a max |dF|, un panel por grupo
    g_all = list(OrderedDict.fromkeys(D["group"]))
    n = len(g_all)
    nc = min(4, n)
    nr = int(np.ceil(n / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.4 * nc, 3.1 * nr), squeeze=False, sharex=True, sharey=True)
    xs = 1e3 * D["kF"] * D["max_sig"]
    ys = 1e3 * D["max_err"]
    lo = max(min(xs.min(), ys.min()) * 0.7, 1e-1)
    hi = max(xs.max(), ys.max()) * 1.4
    small = len(xs) > 500
    for k, g in enumerate(g_all):
        ax = axs[k // nc, k % nc]
        _style(ax)
        m = D["group"] == g
        cal, ev, tr = D["is_cal"], D["ev"], ~D["use"]
        ax.scatter(xs[m & ev], ys[m & ev], s=6 if small else 16, color=SERIES[0], ec="none" if small else "white",
                   lw=0.5, alpha=0.6 if small else 1, label="evaluación", zorder=3)
        if (m & cal).any():
            ax.scatter(xs[m & cal], ys[m & cal], s=6 if small else 16, facecolor="none", ec=INK2,
                       lw=0.5 if small else 0.8, alpha=0.5 if small else 1, label="calibración", zorder=2)
        if (m & tr).any():
            ax.scatter(xs[m & tr], ys[m & tr], s=22, marker="x", color=SERIES[1], lw=1.2,
                       label="en el train", zorder=4)
        ax.plot([lo, hi], [lo, hi], color=MUTED, lw=1, zorder=0)
        ax.axhline(1e3 * err_thr, color=INK2, lw=0.8, ls=":")
        if np.isfinite(D["thr"]):
            ax.axvline(1e3 * D["kF"] * D["thr"], color=INK2, lw=0.8, ls="--")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_title(f"{g} (n = {m.sum()})", loc="left", fontsize=9)
        if k % nc == 0:
            ax.set_ylabel("max |ΔF| (meV/Å)")
        if k // nc == nr - 1:
            ax.set_xlabel("max σ_F calibrada (meV/Å)")
    for k in range(n, nr * nc):
        axs[k // nc, k % nc].set_visible(False)
    axs[0, 0].legend(frameon=False, fontsize=7, loc="upper left", markerscale=1.5)
    fig.text(0.01, 0.005, f"punteada: err_thr = {1e3 * err_thr:.0f} meV/Å · discontinua: umbral de σ (Youden, calibración)",
             fontsize=7, color=INK2)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    _finish(plt, fig, os.path.join(out, "uq_conf_sigma_vs_error.png"))


def _safe(fn):
    return "".join(c if c.isalnum() or c in "-_" else "_" for c in fn)


def plot_traj(out, fn, st, P, D, block_fs):
    plt = _mpl()
    if plt is None:
        return
    idx = st["_idx"]
    t = P["time_fs"][idx] / 1e3
    nbc = P["n_BC_rotos"][idx]
    mag = P["mag_tot"][idx]
    show_bc = np.isfinite(nbc).all() and (nbc > 0).any()
    show_mag = np.isfinite(mag).all() and np.abs(mag).max() > 0.1
    heights = [2.2, 1.2, 1.6] + ([0.8] if show_bc else []) + ([0.9] if show_mag else [])
    fig, axs = plt.subplots(len(heights), 1, figsize=(10, 1.5 * sum(heights) + 0.6), sharex=True,
                            gridspec_kw=dict(height_ratios=heights))
    blocks = st["_blocks"]
    for ax in axs:
        _style(ax)
        for b in blocks:
            if b[3]:
                ax.axvspan(b[0] / 1e3, b[1] / 1e3, color=CALBG, lw=0, zorder=0)
    tr = ~D["use"][idx]

    # 1. fuerzas: max |dF| y k·max sigma
    ax = axs[0]
    ax.plot(t, 1e3 * D["max_err"][idx], color=SERIES[0], lw=0.8, label="max |ΔF| (error real)")
    ax.plot(t, 1e3 * D["kF"] * D["max_sig"][idx], color=SERIES[1], lw=0.8, label="k · max σ_F (comité calibrado)")
    if tr.any():
        ax.scatter(t[tr], 1e3 * D["max_err"][idx][tr], s=14, marker="x", color=INK, lw=1, zorder=5,
                   label="frame del train")
    ax.set_yscale("log")
    ax.set_ylabel("meV/Å")
    ax.set_title(f"{fn}: máximo por frame", loc="left", fontsize=10)
    ax.legend(frameon=False, fontsize=7.5, loc="upper left", ncol=3)

    # 2. RMSE / RMV por bloque
    ax = axs[1]
    if blocks:
        x = np.array([[b[0], b[1]] for b in blocks]).ravel() / 1e3
        y = np.repeat([b[2] for b in blocks], 2)
        ax.plot(x, y, color=INK, lw=1.4)
        ax.axhline(1, color=MUTED, lw=1, ls="--")
        ax.set_ylim(0, max(2, 1.1 * y.max()))
    ax.set_ylabel("RMSE / RMV")
    ax.set_title(f"Calibración por bloque de {block_fs:g} fs (fuerzas por átomo; 1 = calibrado, > 1 = demasiado confiado)",
                 loc="left", fontsize=9)

    # 3. energía: error ± k sigma_E
    ax = axs[2]
    ee = 1e3 * D["e_err"][idx]
    es = 1e3 * D["kE"] * D["e_sig"][idx]
    ax.fill_between(t, -es, es, color=SERIES[1], alpha=0.25, lw=0, label="± k σ_E")
    ax.plot(t, ee, color=SERIES[0], lw=0.8, label="error E (comité − DFT)")
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_ylabel("meV/át")
    ax.set_title("Energía por átomo", loc="left", fontsize=9)
    ax.legend(frameon=False, fontsize=7.5, loc="upper left", ncol=2)

    k = 3
    if show_bc:
        ax = axs[k]; k += 1
        ax.step(t, nbc, where="mid", color=INK2, lw=1)
        ax.set_ylabel("n")
        ax.set_ylim(-0.1, max(1, nbc.max()) + 0.3)
        from matplotlib.ticker import MaxNLocator
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_title("Enlaces B–C rotos (d > 2.2 Å)", loc="left", fontsize=9)
    if show_mag:
        ax = axs[k]; k += 1
        ax.plot(t, mag, color=INK2, lw=0.8)
        ax.set_ylabel("μB")
        ax.set_title("Magnetización total", loc="left", fontsize=9)
    axs[-1].set_xlabel("t (ps)")
    fig.text(0.01, 0.003, "fondo gris: bloques usados para calibrar", fontsize=7, color=INK2)
    fig.tight_layout(rect=(0, 0.015, 1, 1))
    _finish(plt, fig, os.path.join(out, f"uq_trayectoria_{_safe(fn)}.png"))


def informe(P, args, figuras=True):
    """Calibra, valida, escribe CSV/JSON en args.out y dibuja las figuras. Devuelve un dict con todo."""
    os.makedirs(args.out, exist_ok=True)
    rows, summ, cal, rel, relE, D = analyse(P, args)
    tst = trajectory_stats(P, D, args)
    cal["trayectorias"] = {fn: {k: v for k, v in st.items() if not k.startswith("_")} for fn, st in tst.items()}
    write_csv(os.path.join(args.out, "uq_por_config.csv"), rows)
    write_csv(os.path.join(args.out, "uq_resumen.csv"), summ)
    with open(os.path.join(args.out, "uq_calibracion.json"), "w") as fh:
        json.dump(cal, fh, indent=2, ensure_ascii=False, default=float)
    print()
    print_table(summ)
    t = cal["umbral_max_sigmaF_sin_calibrar_eV_A"]
    if t is not None:
        print(f"\nUmbral para MD / aprendizaje activo: max_i sigma_F,i (sin calibrar) >= {1e3 * t:.0f} meV/Å "
              f"detecta en calibración el {100 * cal['umbral_TPR']:.0f} % de las conf. con max|dF| > "
              f"{1e3 * args.err_thr:.0f} meV/Å, con un {100 * cal['umbral_FPR']:.0f} % de falsas alarmas.\n"
              f"  Con MACECalculator(model_paths=[...]), que usa var con ddof=0: sqrt(sum_c forces_var) >= "
              f"{1e3 * cal['umbral_max_sigmaF_convenio_MACECalculator_eV_A']:.0f} meV/Å.")
    else:
        print(f"\nSin umbral: en calibración no hay conf. de las dos clases con err_thr = {args.err_thr} eV/Å")
    print_traj(tst, D["kF"])
    figs = []
    if figuras:
        plots(args.out, rel, relE, D, args.err_thr)
        figs = ["uq_fiabilidad.png", "uq_conf_sigma_vs_error.png"]
        for fn, st in tst.items():
            plot_traj(args.out, fn, st, P, D, args.block_fs)
            figs.append(f"uq_trayectoria_{_safe(fn)}.png")
    print(f"\nSalida en {args.out}/: uq_resumen.csv, uq_por_config.csv, uq_calibracion.json"
          + (", " + ", ".join(figs) if figs else ""))
    return dict(rows=rows, resumen=summ, calibracion=cal, trayectorias=tst, D=D)

## 3. Evaluar el comité

5 modelos × 2500 frames de 126 átomos. Las predicciones se guardan en `<out>/committee_preds.npz` junto con
la lista de modelos y archivos; si no cambia nada, la siguiente vez se leen en lugar de recalcular.

In [ ]:
os.makedirs(args.out, exist_ok=True)
npz = os.path.join(args.out, "committee_preds.npz")
firma = dict(modelos=[os.path.abspath(m) for m in MODELOS], trayectoria=os.path.abspath(TRAYECTORIA),
             calib=CALIB_XYZ, train=TRAIN_XYZ, stride=args.stride, energy_key=args.energy_key,
             forces_key=args.forces_key, head=args.head, dtype=args.dtype)
firma_path = os.path.join(args.out, "committee_preds_firma.json")
reusar = (not RECALCULAR and os.path.exists(npz) and os.path.exists(firma_path)
          and json.load(open(firma_path)) == firma)

if reusar:
    P = load_preds(npz)
    print(f"Predicciones leídas de {npz}: {P['E_pred'].shape[0]} modelos, {P['E_pred'].shape[1]} configuraciones")
else:
    frames = read_sets([TRAYECTORIA], args.energy_key, args.forces_key, args.group_by, "eval", args.stride)
    if CALIB_XYZ:
        frames += read_sets(CALIB_XYZ, args.energy_key, args.forces_key, args.group_by, "calib")
    if TRAIN_XYZ:
        mark_in_train(frames, TRAIN_XYZ, args.energy_key, args.match_tol_E)
    print(f"{len(frames)} configuraciones, {sum(len(f['atoms']) for f in frames)} átomos; "
          f"evaluando {len(MODELOS)} modelos en {args.device}")
    E, F = evaluate(MODELOS, frames, args.device, args.dtype, args.head)
    save_preds(npz, MODELOS, frames, E, F)
    json.dump(firma, open(firma_path, "w"), indent=1)
    P = load_preds(npz)
    print(f"Predicciones guardadas en {npz}")

## 4. Calibración, validación y figuras

Cómo leer el resultado:
- **k** (factor de calibración): k ≫ 1 indica que los 5 modelos se parecen demasiado entre sí (todos parten de
  mace-mh-1) y σ en bruto infraestima el error. Si `musil` y `rmv` difieren mucho, un único factor no basta.
- **Diagrama de fiabilidad**: puntos sobre la diagonal = calibrado; por encima = el comité se fía demasiado.
- **ρ conf** (Spearman por frame) y **AUC**: si σ ordena bien los frames con más error. AUC 0.5 = al azar.
- **Figura temporal**: si el error (azul) y k·σ (naranja) suben a la vez; RMSE/RMV por bloque > 1 donde el comité
  se queda corto; fondo gris = bloques usados para calibrar.

In [ ]:
R = informe(P, args)

## 5. Peores frames

Los 10 frames de evaluación con más error de fuerza, para mirarlos (paso, tiempo, σ calibrada, B–C rotos).

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(R["rows"])
    cols = ["step", "time_fs", "uso", "max_errF_meV_A", "max_sigF_cal_meV_A", "rmse_F_meV_A",
            "err_E_meV_at", "sigma_E_cal_meV_at", "n_BC_rotos", "mag_tot"]
    display(df[df.uso == "evaluacion"].sort_values("max_errF_meV_A", ascending=False)[cols].head(10).round(2))
except ImportError:
    print("Sin pandas: la tabla completa está en", os.path.join(args.out, "uq_por_config.csv"))

## 6. Otra calibración sin volver a evaluar

`split="tiempo"` calibra con la primera mitad de la trayectoria y valida con la segunda: mide si la calibración
aguanta hacia delante en el tiempo (lo que pasaría al vigilar una MD nueva). Para calibrar con el test hay que
poner `CALIB_XYZ` arriba y repetir la sección 3, porque el test no está en las predicciones guardadas.

In [ ]:
args_t = SimpleNamespace(**{**vars(args), "split": "tiempo", "out": args.out + "_tiempo"})
R_t = informe(P, args_t)